Distance functions

In [4]:
import math 

def l2(a,b):
    return sum((ai-bi)**2 for ai,bi in zip(a,b))

def l1(a,b):
    return sum(abs(ai-bi) for ai,bi in zip(a,b))

def cosine_distance(a,b):
    dot_prod=sum((ai*bi) for ai,bi in zip(a,b))
    a_mag=math.sqrt(sum(ai**2 for ai in a))
    b_mag=math.sqrt(sum( bi**2 for bi in b))
    if a_mag == 0 or b_mag == 0:
        return 1.0
    den=a_mag*b_mag
    return 1 -(dot_prod/den)

def minkowski_distance(a, b, p=2):
    if p == float('inf'):
        return max(abs(ai - bi) for ai, bi in zip(a, b))
    return sum(abs(ai - bi) ** p for ai, bi in zip(a, b)) ** (1 / p)


In [5]:
# Test data
a = [1, 2, 3]
b = [4, 5, 6]
c = [0, 0, 0]

print("=== L2 (Euclidean) ===")
print(f"l2({a}, {b}) = {l2(a, b)}")          # 27 (3^2 * 3)
print(f"l2({a}, {a}) = {l2(a, a)}")          # 0
print(f"l2({a}, {c}) = {l2(a, c)}")          # 14 (1+4+9)

print("\n=== L1 (Manhattan) ===")
print(f"l1({a}, {b}) = {l1(a, b)}")          # 9 (3+3+3)
print(f"l1({a}, {a}) = {l1(a, a)}")          # 0

print("\n=== Cosine ===")
print(f"cosine_distance({a}, {b}) = {cosine_distance(a, b):.6f}")
print(f"cosine_distance({a}, {a}) = {cosine_distance(a, a):.6f}")  # ~0
print(f"cosine_distance({a}, {c}) = {cosine_distance(a, c)}")      # 1.0 (zero vector)

print("\n=== Minkowski ===")
print(f"p=1  : {minkowski_distance(a, b, 1)}")       # 9 (same as L1)
print(f"p=2  : {minkowski_distance(a, b, 2)}")       # 27 (same as L2)
print(f"p=3  : {minkowski_distance(a, b, 3)}")       # 3 * 3^(1/3) ≈ 4.3267
print(f"p=inf: {minkowski_distance(a, b, float('inf'))}")  # 3 (max abs diff)

=== L2 (Euclidean) ===
l2([1, 2, 3], [4, 5, 6]) = 27
l2([1, 2, 3], [1, 2, 3]) = 0
l2([1, 2, 3], [0, 0, 0]) = 14

=== L1 (Manhattan) ===
l1([1, 2, 3], [4, 5, 6]) = 9
l1([1, 2, 3], [1, 2, 3]) = 0

=== Cosine ===
cosine_distance([1, 2, 3], [4, 5, 6]) = 0.025368
cosine_distance([1, 2, 3], [1, 2, 3]) = 0.000000
cosine_distance([1, 2, 3], [0, 0, 0]) = 1.0

=== Minkowski ===
p=1  : 9.0
p=2  : 5.196152422706632
p=3  : 4.3267487109222245
p=inf: 3


KNN classifier and regressor

In [7]:
class KNN:
    def __init__(self, k=5, distance_fn=l2, weighted=False,
                 task="classification"):
        self.k = k
        self.distance_fn = distance_fn
        self.weighted = weighted
        self.task = task
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = list(X)
        self.y_train = list(y)

    def predict(self, X):
        return [self._predict_one(x) for x in X]
    
    def _predict_one(self, x):
        distances = []
        for i in range(len(self.X_train)):
            d = self.distance_fn(x, self.X_train[i])
            distances.append((d, self.y_train[i]))
        distances.sort(key=lambda pair: pair[0])
        neighbors = distances[: self.k]

        if self.task == "classification":
            return self._classify(neighbors)
        return self._regress(neighbors)

    def _classify(self, neighbors):
        if self.weighted:
            votes = {}
            for dist, label in neighbors:
                w = 1.0 / (dist + 1e-10)
                votes[label] = votes.get(label, 0) + w
        else:
            votes = {}
            for _, label in neighbors:
                votes[label] = votes.get(label, 0) + 1
        return max(votes, key=votes.get)

    def _regress(self, neighbors):
        if self.weighted:
            w_sum = 0.0
            val_sum = 0.0
            for dist, val in neighbors:
                w = 1.0 / (dist + 1e-10)
                val_sum += w * val
                w_sum += w
            return val_sum / w_sum if w_sum > 0 else 0.0
        return sum(val for _, val in neighbors) / len(neighbors)

    def predict_with_neighbors(self, x):
        distances = []
        for i in range(len(self.X_train)):
            d = self.distance_fn(x, self.X_train[i])
            distances.append((d, i, self.y_train[i]))
        distances.sort(key=lambda t: t[0])
        neighbors = distances[: self.k]
        prediction = self._predict_one(x)
        return prediction, neighbors


In [8]:
def run_tests():
    # ---------- 1D toy data for easy reasoning ----------
    # Labels: 0 for x < 5, 1 for x >= 5
    X_clf = [[1], [2], [3], [4], [6], [7], [8], [9]]
    y_clf = [0, 0, 0, 0, 1, 1, 1, 1]

    # ---------- Continuous targets for regression ----------
    # y = 2x
    X_reg = [[1], [2], [3], [4], [5], [6], [7], [8]]
    y_reg = [2, 4, 6, 8, 10, 12, 14, 16]

    print("=== 1. Basic classification (k=3, unweighted) ===")
    knn = KNN(k=3, distance_fn=l2, weighted=False, task="classification")
    knn.fit(X_clf, y_clf)
    preds = knn.predict([[4.6], [5.4], [0.5], [9.9]])
    print(preds)  # expect: [0, 1, 0, 1]

    print("\n=== 2. Weighted classification (k=3) ===")
    knn_w = KNN(k=3, distance_fn=l2, weighted=True, task="classification")
    knn_w.fit(X_clf, y_clf)
    preds_w = knn_w.predict([[4.6], [5.4]])
    print(preds_w)  # expect: [0, 1] — nearest neighbor dominates

    print("\n=== 3. Regression, unweighted (k=3) ===")
    knn_r = KNN(k=3, distance_fn=l2, task="regression")
    knn_r.fit(X_reg, y_reg)
    preds_r = knn_r.predict([[4.5], [6.5]])
    # x=4.5 -> neighbors 4,5,6 -> y = 8,10,12 -> mean = 10.0
    # x=6.5 -> neighbors 6,7,8 -> y = 12,14,16 -> mean = 14.0
    print(preds_r)  # expect: [10.0, 14.0]

    print("\n=== 4. Regression, weighted (k=3) ===")
    knn_rw = KNN(k=3, distance_fn=l2, weighted=True, task="regression")
    knn_rw.fit(X_reg, y_reg)
    preds_rw = knn_rw.predict([[4.5], [4.0]])
    # x=4.5: weights ~ 1/0.5, 1/0.5, 1/1.5 -> pulls toward 9 (avg of 8,10)
    # x=4.0: exact hit -> should be very close to 8.0
    print(preds_rw)

    print("\n=== 5. k=1 exact match (classification) ===")
    knn1 = KNN(k=1, task="classification")
    knn1.fit(X_clf, y_clf)
    print(knn1.predict([[3], [6]]))  # expect: [0, 1]

    print("\n=== 6. predict_with_neighbors ===")
    knn2 = KNN(k=3, task="classification")
    knn2.fit(X_clf, y_clf)
    pred, neighbors = knn2.predict_with_neighbors([4.6])
    print(f"prediction={pred}")
    print(f"neighbors={neighbors}")
    # expect: prediction=0
    # neighbors ~ [(0.16000000000000014, 3, 0), (0.3600000000000001, 4, 1), (1.96, 2, 0)]
    # (order: index 3 -> x=4, index 4 -> x=6, index 2 -> x=3)

    print("\n=== 7. Different distance functions ===")
    for name, fn in [("L1", l1), ("L2", l2), ("cosine", cosine_distance),
                     ("minkowski p=3", lambda a, b: minkowski_distance(a, b, 3))]:
        m = KNN(k=3, distance_fn=fn, task="classification")
        m.fit(X_clf, y_clf)
        print(f"{name:15s}: {m.predict([[4.6], [5.4]])}")

    print("\n=== 8. Multi-dimensional classification ===")
    X_2d = [[0, 0], [0, 1], [1, 0], [1, 1], [10, 10], [10, 11], [11, 10], [11, 11]]
    y_2d = ["A", "A", "A", "A", "B", "B", "B", "B"]
    m = KNN(k=3, task="classification")
    m.fit(X_2d, y_2d)
    print(m.predict([[0.2, 0.2], [10.5, 10.5], [5, 5]]))
    # expect: ['A', 'B', <tie broken — either 'A' or 'B'>]

    print("\n=== 9. Tie-breaking sanity check (k=2, 1 vote each) ===")
    m = KNN(k=2, task="classification")
    m.fit([[0], [10]], [0, 1])
    # x=5 -> neighbors at distance 5 to both -> tie
    print(m.predict([[5]]))  # either 0 or 1 — deterministic per dict order

    print("\n=== 10. Weighted classification edge: exact match at dist 0 ===")
    m = KNN(k=3, weighted=True, task="classification")
    m.fit(X_clf, y_clf)
    # x=3 is exactly a training point with label 0 -> weight 1/1e-10 dominates
    print(m.predict([[3], [7]]))  # expect: [0, 1]


if __name__ == "__main__":
    run_tests()

=== 1. Basic classification (k=3, unweighted) ===
[0, 1, 0, 1]

=== 2. Weighted classification (k=3) ===
[0, 1]

=== 3. Regression, unweighted (k=3) ===
[8.0, 12.0]

=== 4. Regression, weighted (k=3) ===
[8.84210526310471, 8.0]

=== 5. k=1 exact match (classification) ===
[0, 1]

=== 6. predict_with_neighbors ===
prediction=0
neighbors=[(0.3599999999999996, 3, 0), (1.960000000000001, 4, 1), (2.5599999999999987, 2, 0)]

=== 7. Different distance functions ===
L1             : [0, 1]
L2             : [0, 1]
cosine         : [0, 0]
minkowski p=3  : [0, 1]

=== 8. Multi-dimensional classification ===
['A', 'B', 'A']

=== 9. Tie-breaking sanity check (k=2, 1 vote each) ===
[0]

=== 10. Weighted classification edge: exact match at dist 0 ===
[0, 1]


Feature scaling

In [9]:
def standardize(X):
    n = len(X)
    d = len(X[0])
    means = [sum(X[i][j] for i in range(n)) / n for j in range(d)]
    stds = [
        max(1e-10, (sum((X[i][j] - means[j]) ** 2 for i in range(n)) / n) ** 0.5)
        for j in range(d)
    ]
    return [[((X[i][j] - means[j]) / stds[j]) for j in range(d)] for i in range(n)], means, stds

In [10]:
def test_standardize():
    print("=== 1. Simple 1D column ===")
    X = [[1], [2], [3], [4], [5]]
    Xs, means, stds = standardize(X)
    print("means:", means)   # [3.0]
    print("stds :", stds)    # [sqrt(2) ≈ 1.4142135623]
    print("Xs   :", Xs)
    # Each standardized column should have mean ≈ 0 and std ≈ 1
    n = len(Xs)
    col_mean = sum(r[0] for r in Xs) / n
    col_std = (sum((r[0] - col_mean) ** 2 for r in Xs) / n) ** 0.5
    print(f"col mean={col_mean:.6f}, col std={col_std:.6f}")
    assert abs(col_mean) < 1e-12
    assert abs(col_std - 1.0) < 1e-12

    print("\n=== 2. Two columns, different scales ===")
    X = [[1, 100], [2, 200], [3, 300]]
    Xs, means, stds = standardize(X)
    print("means:", means)  # [2.0, 200.0]
    print("stds :", stds)   # [sqrt(2/3)≈0.8164966, sqrt(20000/3)≈81.64966]
    print("Xs   :", Xs)
    # Both columns should now be standardized
    for j in range(2):
        m = sum(r[j] for r in Xs) / len(Xs)
        s = (sum((r[j] - m) ** 2 for r in Xs) / len(Xs)) ** 0.5
        print(f"col {j}: mean={m:.6f}, std={s:.6f}")
        assert abs(m) < 1e-12
        assert abs(s - 1.0) < 1e-12

    print("\n=== 3. Constant column (zero variance) ===")
    X = [[5, 1], [5, 2], [5, 3]]
    Xs, means, stds = standardize(X)
    print("means:", means)  # [5.0, 2.0]
    print("stds :", stds)   # [1e-10, sqrt(2/3)]
    print("Xs   :", Xs)
    # Constant column: (5-5)/1e-10 = 0.0 — no div-by-zero
    assert all(r[0] == 0.0 for r in Xs)
    assert stds[0] == 1e-10

    print("\n=== 4. Single row ===")
    X = [[7, 8, 9]]
    Xs, means, stds = standardize(X)
    print("means:", means)  # [7.0, 8.0, 9.0]
    print("stds :", stds)   # [1e-10, 1e-10, 1e-10]
    print("Xs   :", Xs)     # [[0.0, 0.0, 0.0]]
    assert Xs == [[0.0, 0.0, 0.0]]

    print("\n=== 5. Negative values ===")
    X = [[-2, -1], [0, 1], [2, 3]]
    Xs, means, stds = standardize(X)
    print("means:", means)  # [0.0, 1.0]
    print("stds :", stds)   # [sqrt(8/3)≈1.63299, sqrt(8/3)≈1.63299]
    print("Xs   :", Xs)
    # Symmetric around zero: first row should be negatives of last row
    for j in range(2):
        assert abs(Xs[0][j] + Xs[2][j]) < 1e-12
        m = sum(r[j] for r in Xs) / 3
        assert abs(m) < 1e-12

    print("\n=== 6. Hand-computed exact check ===")
    X = [[1], [3]]
    Xs, means, stds = standardize(X)
    # mean=2, std=sqrt(((1-2)^2+(3-2)^2)/2)=sqrt(1)=1
    # standardized: [-1.0, 1.0]
    assert means == [2.0]
    assert stds == [1.0]
    assert Xs == [[-1.0], [1.0]]
    print("Exact values OK:", Xs)

    print("\n✅ All standardize tests passed")

test_standardize()

=== 1. Simple 1D column ===
means: [3.0]
stds : [1.4142135623730951]
Xs   : [[-1.414213562373095], [-0.7071067811865475], [0.0], [0.7071067811865475], [1.414213562373095]]
col mean=0.000000, col std=1.000000

=== 2. Two columns, different scales ===
means: [2.0, 200.0]
stds : [0.816496580927726, 81.64965809277261]
Xs   : [[-1.224744871391589, -1.224744871391589], [0.0, 0.0], [1.224744871391589, 1.224744871391589]]
col 0: mean=0.000000, std=1.000000
col 1: mean=0.000000, std=1.000000

=== 3. Constant column (zero variance) ===
means: [5.0, 2.0]
stds : [1e-10, 0.816496580927726]
Xs   : [[0.0, -1.224744871391589], [0.0, 0.0], [0.0, 1.224744871391589]]

=== 4. Single row ===
means: [7.0, 8.0, 9.0]
stds : [1e-10, 1e-10, 1e-10]
Xs   : [[0.0, 0.0, 0.0]]

=== 5. Negative values ===
means: [0.0, 1.0]
stds : [1.632993161855452, 1.632993161855452]
Xs   : [[-1.224744871391589, -1.224744871391589], [0.0, 0.0], [1.224744871391589, 1.224744871391589]]

=== 6. Hand-computed exact check ===
Exact value